In [8]:
import torch
import numpy as np
from torch import nn


class Generator(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(100, 256),
            nn.ReLU(),
            nn.Linear(256, 512),
            nn.ReLU(),
            nn.Linear(512, 1024),
            nn.ReLU(),
            nn.Linear(1024, 784),
            nn.Tanh(),
        )

    def forward(self, x):
        output = self.model(x)
        output = output.view(x.size(0), 1, 28, 28)
        return output


generator_path = "generator_20"
tensor_path = "tensor.pt"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

try:
    generator = torch.load(generator_path, map_location=device, weights_only=False)
except:
    generator = Generator().to(device)
    state = torch.load(generator_path, map_location=device)
    generator.load_state_dict(state)

generator.eval()

z = torch.load(tensor_path, map_location=device)
if z.ndim == 1:
    z = z.unsqueeze(0)

with torch.no_grad():
    generated = generator(z).cpu().numpy()

img = generated[0, 0]

img_min = img.min()
img_max = img.max()
img_255 = (img - img_min) / (img_max - img_min) * 255.0

img_255 = np.floor(img_255 + 0.5).clip(0, 255).astype(np.uint8)

img_255 = 255 - img_255

mean_intensity = img_255.mean()
print(round(float(mean_intensity), 3))

155.867
